In [1]:
import os
import json
import glob
from PIL import Image
from transformers import AutoTokenizer, AutoProcessor

In [2]:
model_id = "google/gemma-3-27b-it"
processor = AutoProcessor.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [3]:
image = Image.open(r"..\person\person\000000006771.jpg")
prompt = "test prompt"
messages = [
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": ""}]}
]
text_prompt = processor.apply_chat_template(messages, tokenize=False)

inputs = processor(text=text_prompt, images=[image], return_tensors="pt")

input_ids = inputs["input_ids"][0]

In [4]:
sum(input_ids == 262144)

tensor(256)

In [5]:
def chunk_book_to_dataset(books_path, output_jsonl_path, chunk_size=256):
    chunks = []
    books = glob.glob(os.path.join(books_path, "*.txt"))
    for book in books:
        print(book)
        with open(book, 'r', encoding='utf-8') as f:
            full_text = f.read()
        
            token_ids = tokenizer.encode(full_text, add_special_tokens=False)
            total_tokens = len(token_ids)

            
            for i in range(0, total_tokens, chunk_size):
                chunk_ids = token_ids[i : i + chunk_size]
                
                if len(chunk_ids) < chunk_size:
                    continue
                    
                chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
                
                chunks.append(chunk_text)

    with open(output_jsonl_path, 'w', encoding='utf-8') as f_out:
        f_out.write(json.dumps(chunks) + '\n')
            
    print("Number of chunks created:", len(chunks))


chunk_book_to_dataset(os.path.join('..', 'books'), os.path.join('..', 'curated_data', 'text', 'text_dataset', 'train_text.jsonl'), chunk_size=256)

..\books\alice_in_the_wonderland.txt
..\books\Anna_Karenina.txt
..\books\bible.txt
..\books\brotherskaramaz00dost_djvu.txt
..\books\crime_and_punishment.txt
..\books\Don_Quixote.txt
..\books\frankestien.txt
..\books\history_of_philosophy.txt
..\books\History_of_Tom_Jones.txt
..\books\Jane_Eyre_An_Autobiography.txt
..\books\les_miserables.txt
..\books\Little_Women.txt
..\books\Middlemarch.txt
..\books\Moby_Dick.txt
..\books\The_Complete_Works_of_William_Shakespeare.txt
..\books\The_Count_of_Monte_Cristo.txt
..\books\The_Mysteries_of_Udolpho.txt
..\books\the_odyssey.txt
..\books\Ulysses.txt
..\books\war_and_peace.txt
Number of chunks created: 40711
